# Generate User Profile Summaries with Mistral-7B
**AdRec-GenAI — RecSys rebuttal / resubmission (ALL 7,176 users)**

This version processes **every user in big_matrix** (7,176), not just the 1,411 small_matrix eval users — needed so B1-Gen/B2-Gen are trained on full-coverage generative embeddings.

Upload these 2 files to Google Drive first:
- `big_matrix.csv`
- `kuairec_caption_category.csv`

Then set your Drive folder path in Cell 2 and run all cells.

**Runtime:** ~4-8 h on a T4 for ~5,800 remaining users (progress is cached every 50 users, so you can stop and resume — already-done users are skipped). Upload your existing `user_generative_summaries.json` to the output folder to keep the 1,411 users already generated.

In [ ]:
# ── Cell 1: Install ────────────────────────────────────────────────────────────
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf
print('Done')

In [ ]:
# ── Cell 2: Mount Drive + set paths ───────────────────────────────────────────
# !! CHANGE these paths to match where YOUR files are in Drive !!
from google.colab import drive
drive.mount('/content/drive')

import os

# Change this to wherever you uploaded the data files
DATA_DIR = '/content/drive/MyDrive/Colab Notebooks/AdRec-GenAI/data'
OUT_DIR  = '/content/drive/MyDrive/Colab Notebooks/AdRec-GenAI/embeddings'
os.makedirs(OUT_DIR, exist_ok=True)

print('Checking files...')
all_ok = True
for f in ['big_matrix.csv', 'kuairec_caption_category.csv']:
    path = os.path.join(DATA_DIR, f)
    if os.path.exists(path):
        size = os.path.getsize(path) // (1024*1024)
        print(f'  ✅ {f} ({size} MB)')
    else:
        print(f'  ❌ {f} NOT FOUND at {path}')
        all_ok = False

if not all_ok:
    print('\n⚠️  Fix the DATA_DIR path above before continuing')
else:
    print('\nAll files found ✅')

In [ ]:
# ── Cell 3: Load model ─────────────────────────────────────────────────────────
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import traceback

MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'

print(f'GPU : {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Loading {MODEL_ID}...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model.eval()
print('Model loaded ✅')

In [ ]:
# ── Cell 4: DIAGNOSTIC — test generation on ONE prompt ────────────────────────
# Run this cell first. If it fails, read the full error before proceeding.
import traceback

TEST_PROMPT = 'Write one sentence about cats.'
print(f'Testing with: "{TEST_PROMPT}"')

try:
    inputs = tokenizer(TEST_PROMPT, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False,        # greedy — most reliable
            pad_token_id=tokenizer.eos_token_id,
        )
    result = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    print(f'\n✅ Generation works!')
    print(f'Output: {result}')
except Exception:
    print('\n❌ Generation FAILED. Full error:')
    traceback.print_exc()

In [ ]:
# ── Cell 5: Build raw user profiles ───────────────────────────────────────────
import pandas as pd
from tqdm.notebook import tqdm

TOP_K       = 20
ITEM_SEP    = '; '
DEDUPLICATE = True
FALLBACK    = 'no description'
FIELD_SEP   = ' | '
FIELDS = [
    ('manual_cover_text',          'cover'),
    ('caption',                    'caption'),
    ('topic_tag',                  'topics'),
    ('first_level_category_name',  'category1'),
    ('second_level_category_name', 'category2'),
]

def build_text(row):
    parts = [f'{p}: {row[c].strip()}' for c, p in FIELDS
             if c in row and isinstance(row[c], str) and row[c].strip()]
    return FIELD_SEP.join(parts) if parts else FALLBACK

print('Loading item metadata...')
meta_df = pd.read_csv(os.path.join(DATA_DIR, 'kuairec_caption_category.csv'),
                      sep=None, engine='python', on_bad_lines='skip')
meta_df.columns = [c.strip() for c in meta_df.columns]
vid_col = next(c for c in ['video_id','item_id','id'] if c in meta_df.columns)
meta_df[vid_col] = pd.to_numeric(meta_df[vid_col], errors='coerce')
meta_df = meta_df.dropna(subset=[vid_col]).copy()
meta_df[vid_col] = meta_df[vid_col].astype(int)
id2text = {int(row[vid_col]): build_text(row) for _, row in meta_df.iterrows()}
print(f'  {len(id2text):,} items')

print('Loading big_matrix (ALL users)...')
big = pd.read_csv(
    os.path.join(DATA_DIR, 'big_matrix.csv'),
    dtype={'user_id':'int32', 'video_id':'int32'},
    usecols=['user_id', 'video_id', 'watch_ratio'],
)
print(f'  {len(big):,} interactions for {big["user_id"].nunique():,} users')

print('Building raw profiles...')
user_ids, raw_texts = [], []
for uid, grp in tqdm(big.sort_values('watch_ratio', ascending=False).groupby('user_id')):
    top_items = grp['video_id'].head(TOP_K).tolist()
    descs = [id2text.get(int(v), FALLBACK) for v in top_items]
    if DEDUPLICATE:
        seen, unique = set(), []
        for d in descs:
            if d not in seen: seen.add(d); unique.append(d)
        descs = unique
    user_ids.append(int(uid))
    raw_texts.append(ITEM_SEP.join(descs))
print(f'✅ {len(user_ids):,} profiles built')

In [ ]:
# ── Cell 6: CLEAR OLD BAD CACHE (run once if you had all-error run before) ────
import json

CACHE_PATH = os.path.join(OUT_DIR, 'user_generative_summaries.json')

if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        old = json.load(f)
    old_count = len(old.get('summaries', {}))
    # Detect bad cache: if all summaries are short (raw fallback text, not real summaries)
    samples = list(old.get('summaries', {}).values())[:20]
    avg_len = sum(len(s) for s in samples) / max(len(samples), 1)
    print(f'Existing cache: {old_count} entries, avg length: {avg_len:.0f} chars')
    if avg_len < 150:   # real summaries are ~200-400 chars; fallback is raw text ~100-200
        print('⚠️  Cache looks like it contains fallback text, not real summaries.')
        print('   Deleting bad cache so we start fresh...')
        os.remove(CACHE_PATH)
        print('   Deleted ✅')
    else:
        print('Cache looks valid — will resume from it.')
else:
    print('No existing cache — starting fresh.')

In [ ]:
# ── Cell 7: Generate summaries ─────────────────────────────────────────────────
import json, traceback

MAX_NEW_TOKENS = 80
SAVE_EVERY     = 50

PROMPT_TEMPLATE = (
    'You are analyzing a short-video platform user. '
    'Based on the following videos they watched most, write a 2-sentence '
    'summary of their interests in plain English. Be concise.\n'
    'Videos: {raw_profile}\nSummary:'
)

# Load or create cache
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        cache = json.load(f)
    print(f'Resuming: {len(cache["summaries"]):,} users already done')
else:
    cache = {'model': MODEL_ID, 'backend': 'colab_transformers',
             'generated_at': pd.Timestamp.utcnow().isoformat(),
             'summaries': {}}

summaries = cache['summaries']
remaining = [(uid, txt) for uid, txt in zip(user_ids, raw_texts)
             if str(uid) not in summaries]
print(f'{len(remaining):,} users to process')

def generate_summary(raw_profile):
    """Generate using direct tokenization (no chat template — more robust)."""
    prompt = PROMPT_TEMPLATE.format(raw_profile=raw_profile[:500])  # cap input length
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=512,
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,                    # greedy = faster + reproducible
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(
        out[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()
    # Clean up any repeated prompt artifacts
    if 'Summary:' in decoded:
        decoded = decoded.split('Summary:')[-1].strip()
    return decoded

# Test on first user before full run
print('\nTesting on first user...')
try:
    test_summary = generate_summary(raw_texts[0])
    print(f'✅ Test passed: {test_summary[:100]}...')
except Exception:
    print('❌ Test failed — full error below. Fix before running all users.')
    traceback.print_exc()
    raise

# Main generation loop
print(f'\nGenerating {len(remaining):,} summaries...')
errors = 0
first_error_shown = False

for i, (uid, raw_text) in enumerate(tqdm(remaining, desc='Generating')):
    try:
        summaries[str(uid)] = generate_summary(raw_text)
    except Exception as e:
        errors += 1
        summaries[str(uid)] = raw_text[:200]  # fallback
        if not first_error_shown:
            print(f'\nFirst error (user {uid}):')
            traceback.print_exc()              # show FULL error once
            first_error_shown = True

    if (i + 1) % SAVE_EVERY == 0:
        with open(CACHE_PATH, 'w') as f:
            json.dump(cache, f, indent=2, ensure_ascii=False)
        print(f'  Saved {i+1}/{len(remaining)}')

with open(CACHE_PATH, 'w') as f:
    json.dump(cache, f, indent=2, ensure_ascii=False)

print(f'\n✅ {len(summaries):,} summaries saved')
print(f'   Errors: {errors}/{len(remaining)}')

In [ ]:
# ── Cell 8: Verify ────────────────────────────────────────────────────────────
print(f'Total: {len(summaries):,} summaries')
print(f'File:  {CACHE_PATH}')
print(f'Size:  {os.path.getsize(CACHE_PATH)/1024:.0f} KB\n')

# Show 3 examples
for uid_str, summary in list(summaries.items())[:3]:
    print(f'User {uid_str}:')
    print(f'  {summary}')
    print(f'  (length: {len(summary)} chars)\n')

# Quality check — real summaries should be > 100 chars
avg = sum(len(s) for s in summaries.values()) / len(summaries)
print(f'Avg summary length: {avg:.0f} chars')
if avg < 100:
    print('⚠️  Summaries look too short — may still be fallback text')
else:
    print('✅ Summaries look good!')

## Done — next steps on your Mac

**1. Download** `user_generative_summaries.json` from Drive

**2. Put it here on your Mac:**
```
AdRec-GenAI/kuairec/embeddings/user_generative_summaries.json
```

**3. Rebuild the generative user embeddings locally:**
```bash
cd /Users/tanushreenepal/Desktop/AdRec-GenAI
python LLM-rec/src/build_user_llm_embeddings.py --use_generative
```

**4. Upload the refreshed `user_llm_embeddings_generative.npy` to Drive**, then run `colab_train_transformers.ipynb` for the transformer models (A2/B2/B2-Gen). The MLPs (A1/B1/B1-Gen) train locally:
```bash
python LLM-rec/src/run_all.py --models a1 b1 b1_gen
```